In [4]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window


def get_spark(cores=8, memory="60g"):
    spark = (
        SparkSession.builder.master(f"local[{cores}]")
        .config("spark.driver.memory", memory)
        .getOrCreate()
    )
    return spark


trec_root = Path("~/scratch/trec-tot-2025").expanduser()
dataset_root = trec_root / "data/enwiki/processed"
gdrive_path = trec_root / "data/gdrive/data"
official_path = trec_root / "data/official"
output_path = trec_root / "results/rerank"

spark = get_spark()

score_root = (dataset_root / "centrality/v1/pagerank.parquet").as_posix()
display(score_root)
score = spark.read.parquet(score_root).withColumnRenamed("node_id", "id")
score.show(n=5)

'/storage/home/hcoda1/8/amiyaguchi3/scratch/trec-tot-2025/data/enwiki/processed/centrality/v1/pagerank.parquet'

+---+--------------------+
| id|            pagerank|
+---+--------------------+
|  0|1.578054438775196...|
|  1|1.578054438775196...|
|  2|1.578054438775196...|
|  3|1.578054438775196...|
|  4|1.578054438775196...|
+---+--------------------+
only showing top 5 rows


In [2]:
cols = ["qid", "Q0", "docid", "rank", "score", "run_name"]
run = spark.read.csv(
    f"{gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run",
    sep="\t",
    header=False,
    inferSchema=True,
).toDF(*cols)
run.show(n=5)
run.printSchema()

+----+---+--------+----+-----+---------------+
| qid| Q0|   docid|rank|score|       run_name|
+----+---+--------+----+-----+---------------+
|2001| Q0| 1752781|   0|  5.0|gemini-25_alias|
|2001| Q0|   58865|   1|  4.0|gemini-25_alias|
|2001| Q0| 4907816|   2|  4.0|gemini-25_alias|
|2001| Q0|49363330|   3|  4.0|gemini-25_alias|
|2001| Q0|73331355|   4|  4.0|gemini-25_alias|
+----+---+--------+----+-----+---------------+
only showing top 5 rows
root
 |-- qid: integer (nullable = true)
 |-- Q0: string (nullable = true)
 |-- docid: integer (nullable = true)
 |-- rank: integer (nullable = true)
 |-- score: double (nullable = true)
 |-- run_name: string (nullable = true)



In [5]:
score_col = "pagerank"
reranked = (
    run.join(score, run.docid == score.id, "inner")
    .select(
        "qid",
        "Q0",
        "docid",
        F.row_number()
        .over(Window.partitionBy("qid").orderBy(F.desc(score_col), "rank"))
        .alias("rank"),
        F.col(score_col).alias("score"),
        F.col("run_name"),
    )
    .orderBy("qid", "rank")
)
reranked.show(n=5)
reranked_path = "/tmp/reranked.run"
Path(reranked_path).parent.mkdir(parents=True, exist_ok=True)
reranked.toPandas().to_csv(
    reranked_path,
    sep=" ",
    index=False,
    header=False,
)
! head {reranked_path}

+----+---+-------+----+--------------------+---------------+
| qid| Q0|  docid|rank|               score|       run_name|
+----+---+-------+----+--------------------+---------------+
|2001| Q0|1752781|   1|1.578054438775196...|gemini-25_alias|
|2001| Q0|  58865|   2|1.578054438775196...|gemini-25_alias|
|2001| Q0|4907816|   3|1.578054438775196...|gemini-25_alias|
|2001| Q0|1650010|   4|1.578054438775196...|gemini-25_alias|
|2002| Q0|1203157|   1|1.578054438775196...|gemini-25_alias|
+----+---+-------+----+--------------------+---------------+
only showing top 5 rows
2001 Q0 1752781 1 1.5780544387751962e-07 gemini-25_alias
2001 Q0 58865 2 1.5780544387751962e-07 gemini-25_alias
2001 Q0 4907816 3 1.5780544387751962e-07 gemini-25_alias
2001 Q0 1650010 4 1.5780544387751962e-07 gemini-25_alias
2002 Q0 1203157 1 1.5780544387751962e-07 gemini-25_alias
2002 Q0 1406169 2 1.5780544387751962e-07 gemini-25_alias
2002 Q0 2199665 3 1.5780544387751962e-07 gemini-25_alias
2002 Q0 1829527 4 1.5780544387

In [6]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    {official_path}/dev3-2025-qrel.txt \
    {gdrive_path}/shared_retrieval_results/gemini-2.5-flash/dev3.run

recip_rank            	all	0.4001
recall_1000           	all	0.5871
ndcg_cut_10           	all	0.4357
ndcg_cut_1000         	all	0.4436


In [8]:
! trec_eval \
    -m ndcg_cut.10,1000 \
    -m recall.1000 \
    -m recip_rank \
    {official_path}/dev3-2025-qrel.txt \
    {reranked_path} 

recip_rank            	all	0.1764
recall_1000           	all	0.3504
ndcg_cut_10           	all	0.2147
ndcg_cut_1000         	all	0.2180
